In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset
from datasets import load_dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from huggingface_hub import hf_hub_download

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Loading synthetic data for labeling: to preserve anonymity, the links to Hugging Face have been removed, but a portion of 50k samples from each dataset is included.

In [ ]:
no_Labels = 2

df_2_label = pd.read_csv('data/2_label.csv')
df_3_label = pd.read_csv('data/3_label.csv')

In [ ]:
# dataset_2 = load_dataset("")
# df_2_label = pd.DataFrame(dataset_2["2_labels"])

# dataset_3 = load_dataset("")
# df_3_label = pd.DataFrame(dataset_3["3_labels"])

# df_2_label.sample(n=50000, random_state=42).to_csv("df_2_label.csv", index=False)
# df_3_label.sample(n=50000, random_state=42).to_csv("df_3_label.csv", index=False)

In [10]:
print("Total size of 2 labels data: ",df_2_label.shape[0])
print("Language counts ",df_2_label['language'].value_counts())

Total size of 2 labels data:  240647
Language counts  language
deu    125617
eng    108375
vie      6655
Name: count, dtype: int64


In [11]:
print("Total size of 3 labels data: ",df_3_label.shape[0])
print("Language counts ",df_3_label['language'].value_counts())


Total size of 3 labels data:  350364
Language counts  language
eng    218092
deu    125617
vie      6655
Name: count, dtype: int64


In [9]:
mstral7b = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
gemma9b = "unsloth/gemma-2-9b-it-bnb-4bit"
qwen14b = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
llama8B = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"



# Labeling for 3 label: Hate / Offensive / Neutral. LightGBM and Mean Average 

In [19]:
columns = [mstral7b, gemma9b, qwen14b]
all_cols = []
for col in columns:
    all_cols.append(col + "_label_1")
    all_cols.append(col + "_label_2")
    all_cols.append(col + "_label_3")
X = df_3_label[all_cols]

with open('3_lgb_label.pkl', 'rb') as f:
    lgb_model = pickle.load(f)
    
val_preds = lgb_model.predict(X, num_iteration=lgb_model.best_iteration)
val_preds = [list(x).index(max(x)) for x in val_preds]
df_3_label["Lgb"] = val_preds
df_3_label["Lgb"].value_counts()

Lgb
2    330908
1     19246
0       210
Name: count, dtype: int64

In [14]:


model_list = [mstral7b, gemma9b, qwen14b]
columns_name = ["_label_1", "_label_2", "_label_3"]

col_pred = []
for col in columns_name:
    col_model = []
    for model_id in model_list:
        col_model.append(model_id + col)
    
    mean_value = df_3_label[col_model].mean(axis=1)
    df_3_label[col + "_mean"] = mean_value
    col_pred.append(col + "_mean")


max_index = df_3_label[col_pred].idxmax(axis=1).apply(lambda x: col_pred.index(x))
df_3_label["Mean"] = max_index + 1
df_3_label["Mean"].value_counts()

Mean
3    307293
2     40747
1      2324
Name: count, dtype: int64

# Labeling for 2 labels: Hate  / Neutral. LightGBM, Vote, Mean Average

In [15]:
columns = [qwen14b, llama8B, mstral7b, gemma9b]

LightGbm

In [18]:


with open('2_lgb_label_1.pkl', 'rb') as f:
    lgb_label_1 = pickle.load(f)

c1 = [x + "_label_1" for x in columns]
X = df_2_label[c1]
val_preds = lgb_label_1.predict(X, num_iteration=lgb_model.best_iteration)
df_2_label["lgb_label_1"] = val_preds



with open('2_lgb_label_2.pkl', 'rb') as f:
    lgb_label_2 = pickle.load(f)

c2 = [x + "_label_2" for x in columns]
X = df_2_label[c2]
val_preds = lgb_label_2.predict(X, num_iteration=lgb_model.best_iteration)
df_2_label["lgb_label_2"] = val_preds



index_1 = df_2_label['lgb_label_1'] > df_2_label['lgb_label_2']
index_2 = df_2_label['lgb_label_1'] > .0

index_all = index_1 & index_2

df_2_label['Lgb'] = 2
df_2_label.loc[index_all, 'Lgb'] = 1
df_2_label['Lgb'].value_counts()

Lgb
2    237883
1      2764
Name: count, dtype: int64

Voting

In [11]:
index_qwen = df_2_label[qwen14b + "_label_1"] > .5
index_llama8b = df_2_label[llama8B+ "_label_1"] > .5
index_gemma9b = df_2_label[gemma9b+ "_label_1"] > .5
index_mstral7b = df_2_label[mstral7b+ "_label_1"] > .5

from collections import Counter

index_all = index_mstral7b.astype(int) + index_gemma9b.astype(int) + index_llama8b.astype(int) + index_qwen.astype(int) 
index_all = index_all >= 2

df_2_label['Vote'] = 2
df_2_label.loc[index_all, 'Vote'] = 1
df_2_label['Vote'].value_counts()

Vote
2    235940
1      4707
Name: count, dtype: int64

Mean Average

In [12]:
c1 = [x + "_label_1" for x in columns]
df_2_label["mean_label_1"] = df_2_label[c1].mean(axis=1)

c2 = [x + "_label_2" for x in columns]
df_2_label["mean_label_2"] = df_2_label[c2].mean(axis=1)

index_1 = df_2_label['mean_label_1'] > df_2_label['mean_label_2']
index_2 = df_2_label['mean_label_1'] > .0
index_all = index_1 & index_2

df_2_label['Mean'] = 2
df_2_label.loc[index_all, 'Mean'] = 1
df_2_label['Mean'].value_counts()

Mean
2    236660
1      3987
Name: count, dtype: int64